# 套件

In [1]:
import os 
import pandas as pd


## 數據觀察與定義

In [2]:
pa_record = pd.read_csv('pa_record.csv', encoding='utf-8')
pa_record.head()

,gameNo,awayTeam,homeTeam,team_type,inning,scored,batterName,batterHand,pitcherName,pitcherHand,...,homeScores,strikes,balls,outs,bases,result,RBI,locationCode,trajectory,hardness
0,1,樂天桃猿,味全龍,away,1,False,陳晨威,L,徐若熙,R,...,0,0,1,0,0,GO,0,4M,G,M
1,1,樂天桃猿,味全龍,away,1,False,林立,R,徐若熙,R,...,0,2,2,1,0,SO,0,NaN,NaN,NaN
2,1,樂天桃猿,味全龍,away,1,False,梁家榮,L,徐若熙,R,...,0,2,1,2,0,2B,0,9LF,G,H
3,1,樂天桃猿,味全龍,away,1,False,廖健富,L,徐若熙,R,...,0,0,1,2,2,1B,0,8,G,H
4,1,樂天桃猿,味全龍,away,2,False,朱育賢,L,徐若熙,R,...,0,2,0,0,0,FO,0,7LSF,F,M


In [3]:
print(pa_record['result'].unique(), len(pa_record['result'].unique()))

['GO' 'SO' '2B' '1B' 'FO' 'E' 'uBB' 'GIDP' 'FC' 'SH' 'SF_E' 'HBP' 'HR'
 'SF' 'SH_E' '3B' 'IBB' nan 'SH_FC' 'GIDP_E' 'IH' 'H' 'ID'] 23


In [4]:
pa_record.iloc[19125:19130, ]

,gameNo,awayTeam,homeTeam,team_type,inning,scored,batterName,batterHand,pitcherName,pitcherHand,...,homeScores,strikes,balls,outs,bases,result,RBI,locationCode,trajectory,hardness
19125,249,中信兄弟,台鋼雄鷹,home,7,True,吳念庭,L,謝榮豪,R,...,4,0,0,1,7,1B,1,34D,L,M
19126,249,中信兄弟,台鋼雄鷹,home,7,True,藍寅倫,L,謝榮豪,R,...,5,2,3,1,7,1B,2,6D,L,M
19127,249,中信兄弟,台鋼雄鷹,home,7,False,張肇元,R,謝榮豪,R,...,7,1,0,1,5,ID,0,15,G,S
19128,249,中信兄弟,台鋼雄鷹,home,7,False,郭永維,R,謝榮豪,R,...,7,2,1,2,5,FC,0,56,G,S
19129,249,中信兄弟,台鋼雄鷹,home,8,False,王博玄,L,陳柏均,L,...,7,1,0,0,0,1B,0,4,G,S


In [ ]:
RESULT_MAPPING = {
    # 安打類
    '1B': '一壘安打 (Single)',
    '2B': '二壘安打 (Double)', 
    '3B': '三壘安打 (Triple)',
    'HR': '全壘打 (Home Run)',
    'H': '安打 (Hit)',
    'IH': '場內全壘打 (Inside-the-Park Home Run)',

    # 出局類
    'GO': '滾地球出局 (Ground Out)',
    'FO': '飛球出局 (Fly Out)',
    'SO': '三振 (Strike Out)',
    'GIDP': '滾地雙殺 (Ground Into Double Play)',    
    'FC': '野手選擇 (Fielder\'s Choice)',    
    'SH_FC': '犧牲觸擊野手選擇 (Sacrifice Hit Fielder\'s Choice)',   

    # 四壞球類
    'uBB': '四壞球 (Unintentional Walk)',
    'IBB': '故意四壞球 (Intentional Walk)',
    'HBP': '觸身球 (Hit By Pitch)',
    
    # 犧牲類
    'SF': '高飛犧牲打 (Sacrifice Fly)',
    'SH': '犧牲觸擊 (Sacrifice Hit/Bunt)',
    
    # 失誤類
    'E': '失誤 (Error)',
    'GIDP_E': '滾地雙殺失誤 (Ground Into Double Play Error)',    

    'SF_E': '高飛犧牲打失誤 (Sacrifice Fly Error)',
    'SH_E': '犧牲觸擊失誤 (Sacrifice Hit Error)',    
    
    
    # 其他
    'ID': '內野高飛球 (Infield fly out)',

}

# 計算打數的內容
At_Bat = ['1B', '2B', '3B', 'HR', 'H', 'GO', 'FO', 'SO', 'E', 'FC', 'GIDP', 'GIDP_E', 'SF_E', 'SH_E', 'SH_FC', 'ID', 'IH']
Hit = ['1B', '2B', '3B', 'HR', 'H', 'IH']
On_Base = ['1B', '2B', '3B', 'HR', 'H', 'uBB', 'IBB', 'HBP']
At_Bat_and_On_Base = At_Bat + ['uBB', 'IBB', 'HBP', 'SF']

# 打擊率：安打數 / 打數 -> Hit / At_Bat
# 上壘率：上壘數 / 打數＋四死＋高飛犧牲打 -> On_Base / (At_Bat + uBB + IBB + HBP + SF)
# 長打率：長打數 / 打數 -> (1B + 2B * 2 + 3B * 3 + HR * 4) / At_Bat
# OPS：打擊率＋長打率 -> Batting_Average + Slugging_Percentage
# 純長打率：長打率－打擊率 -> Slugging_Percentage - Batting_Average
# 三振率：三振數 / 打席 -> SO / PA
# 保送率：保送數 / 打席 -> BB / PA
# 場內安打率：所有安打－全壘打 / 打數－三振－全壘打＋高飛犧牲打
TB = {'1B': 1, '2B': 2, '3B': 3, 'HR': 4, 'IH': 4, 'H': 1}

## 聯盟整體打擊概況

In [6]:
# 打擊率
League_Batting_Average = round(len(pa_record.loc[pa_record['result'].isin(Hit),]) / len(pa_record.loc[pa_record['result'].isin(At_Bat)]), 4)
# 上壘率
League_On_Base_Percentage = round(len(pa_record.loc[pa_record['result'].isin(On_Base)]) / len(pa_record.loc[pa_record['result'].isin(At_Bat_and_On_Base)]), 4)
# 長打率
Total_Bases_Sum = pa_record['result'].apply(lambda n: TB.get(n, 0)).sum()
League_Slugging_Percentage = round(Total_Bases_Sum / len(pa_record.loc[pa_record['result'].isin(At_Bat)]), 4)
# 純長打率
League_Isolated_Power = round(League_Slugging_Percentage - League_Batting_Average, 4)
# 三振率
League_Strikeout_Rate = round(len(pa_record[pa_record['result'] == 'SO']) / len(pa_record), 4)
# 保送率
League_Walk_Rate = round(len(pa_record.loc[pa_record['result'] == 'uBB',]) / len(pa_record), 4)
# 場內安打率
League_Batting_Aberage_on_Balls_In_play = round((len(pa_record.loc[pa_record['result'].isin(Hit)]) - len(pa_record.loc[pa_record['result'] == 'HR',]) )/ (len(pa_record.loc[pa_record['result'].isin(At_Bat)]) - len(pa_record.loc[pa_record['result'] == 'SO',]) - len(pa_record.loc[pa_record['result'] == 'HR',]) + len(pa_record.loc[pa_record['result'] == 'SF',])), 4)
# OPS
League_OPS = round(League_On_Base_Percentage + League_Slugging_Percentage, 4)


print(f"聯盟打擊率：{League_Batting_Average}")
print(f"聯盟上壘率：{League_On_Base_Percentage}")
print(f"聯盟長打率：{League_Slugging_Percentage}")
print(f"聯盟純長打率：{League_Isolated_Power}")
print(f"聯盟三振率：{League_Strikeout_Rate}")
print(f"聯盟保送率：{League_Walk_Rate}")
print(f"聯盟場內安打率：{League_Batting_Aberage_on_Balls_In_play}")
print(f"聯盟 OPS ：{League_OPS}")


聯盟打擊率：0.2637
聯盟上壘率：0.3297
聯盟長打率：0.3617
聯盟純長打率：0.098
聯盟三振率：0.1711
聯盟保送率：0.0773
聯盟場內安打率：0.3109
聯盟 OPS ：0.6914


## 投打對決概況

In [7]:
left_pitcher_verus_left_hitter = pa_record.loc[(pa_record['pitcherHand'] == 'L' ) & (pa_record['batterHand'] == 'L')]
left_pitcher_verus_right_hitter = pa_record.loc[(pa_record['pitcherHand'] == 'L' ) & (pa_record['batterHand'] == 'R')]
right_pitcher_verus_left_hitter = pa_record.loc[(pa_record['pitcherHand'] == 'R' ) & (pa_record['batterHand'] == 'L')]
right_pitcher_verus_right_hitter = pa_record.loc[(pa_record['pitcherHand'] == 'R' ) & (pa_record['batterHand'] == 'R')]



# 左投對左打
left_vs_left_Batting_Average = round(len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(Hit),]) / len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(At_Bat)]), 4)
left_vs_left_On_Base_Percentage = round(len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(On_Base)]) / len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(At_Bat_and_On_Base)]), 4)
# 長打率：長打數 / 打數 -> (1B + 2B * 2 + 3B * 3 + HR * 4 + IH * 4) / At_Bat
left_vs_left_Total_Bases_Sum = left_pitcher_verus_left_hitter['result'].apply(lambda n: TB.get(n, 0)).sum()
left_vs_left_Slugging_Percentage = round(left_vs_left_Total_Bases_Sum / len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(At_Bat)]), 4)
# 純長打率
left_vs_left_Isolated_Power = round(left_vs_left_Slugging_Percentage - left_vs_left_Batting_Average, 4)
# 三振率
left_vs_left_Strikeout_Rate = round(len(left_pitcher_verus_left_hitter[left_pitcher_verus_left_hitter['result'] == 'SO']) / len(left_pitcher_verus_left_hitter), 4)
# 保送率
left_vs_left_Walk_Rate = round(len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'] == 'uBB',]) / len(left_pitcher_verus_left_hitter), 4)
# 場內安打率
left_vs_left_Batting_Aberage_on_Balls_In_play = round((len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(Hit)]) - len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'] == 'HR',]) )/ (len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'].isin(At_Bat)]) - len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'] == 'SO',]) - len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'] == 'HR',]) + len(left_pitcher_verus_left_hitter.loc[left_pitcher_verus_left_hitter['result'] == 'SF',])), 4)


# OPS
left_vs_left_OPS = round(left_vs_left_On_Base_Percentage + left_vs_left_Slugging_Percentage, 4)


print(f"左投對左打打擊率：{left_vs_left_Batting_Average}")
print(f"左投對左打上壘率：{left_vs_left_On_Base_Percentage}")
print(f"左投對左打長打率：{left_vs_left_Slugging_Percentage}")
print(f"左投對左打純長打率：{left_vs_left_Isolated_Power}")
print(f"左投對左打三振率：{left_vs_left_Strikeout_Rate}")
print(f"左投對左打保送率：{left_vs_left_Walk_Rate}")
print(f"左投對左打場內安打率：{left_vs_left_Batting_Aberage_on_Balls_In_play}")
print(f"左投對左打 OPS ：{left_vs_left_OPS}")

左投對左打打擊率：0.2673
左投對左打上壘率：0.3308
左投對左打長打率：0.348
左投對左打純長打率：0.0807
左投對左打三振率：0.1755
左投對左打保送率：0.0714
左投對左打場內安打率：0.3213
左投對左打 OPS ：0.6788


In [8]:
# 左投對右打
left_vs_right_Batting_Average = round(len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(Hit),]) / len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(At_Bat)]), 4)
left_vs_right_On_Base_Percentage = round(len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(On_Base)]) / len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(At_Bat_and_On_Base)]), 4)

left_vs_right_Total_Bases_Sum = left_pitcher_verus_right_hitter['result'].apply(lambda n: TB.get(n, 0)).sum()
left_vs_right_Slugging_Percentage = round(left_vs_right_Total_Bases_Sum / len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(At_Bat)]), 4)


# 純長打率
left_vs_right_Isolated_Power = round(left_vs_right_Slugging_Percentage - left_vs_right_Batting_Average, 4)
# 三振率
left_vs_right_Strikeout_Rate = round(len(left_pitcher_verus_right_hitter[left_pitcher_verus_right_hitter['result'] == 'SO']) / len(left_pitcher_verus_right_hitter), 4)
# 保送率
left_vs_right_Walk_Rate = round(len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'] == 'uBB',]) / len(left_pitcher_verus_right_hitter), 4)
# 場內安打率
left_vs_right_Batting_Aberage_on_Balls_In_play = round((len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(Hit)]) - len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'] == 'HR',]) )/ (len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'].isin(At_Bat)]) - len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'] == 'SO',]) - len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'] == 'HR',]) + len(left_pitcher_verus_right_hitter.loc[left_pitcher_verus_right_hitter['result'] == 'SF',])), 4)


left_vs_right_OPS = round(left_vs_right_On_Base_Percentage + left_vs_right_Slugging_Percentage, 4)


print(f"左投對右打打擊率：{left_vs_right_Batting_Average}")
print(f"左投對右打上壘率：{left_vs_right_On_Base_Percentage}")
print(f"左投對右打長打率：{left_vs_right_Slugging_Percentage}")
print(f"左投對右打純長打率：{left_vs_right_Isolated_Power}")
print(f"左投對右打三振率：{left_vs_right_Strikeout_Rate}")
print(f"左投對右打保送率：{left_vs_right_Walk_Rate}")
print(f"左投對右打場內安打率：{left_vs_right_Batting_Aberage_on_Balls_In_play}")
print(f"左投對右打 OPS ：{left_vs_right_OPS}")


左投對右打打擊率：0.2589
左投對右打上壘率：0.3283
左投對右打長打率：0.3556
左投對右打純長打率：0.0967
左投對右打三振率：0.1584
左投對右打保送率：0.0853
左投對右打場內安打率：0.2992
左投對右打 OPS ：0.6839


In [9]:
# 右投對左打
right_vs_left_Batting_Average = round(len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(Hit),]) / len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(At_Bat)]), 4)
right_vs_left_On_Base_Percentage = round(len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(On_Base)]) / len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(At_Bat_and_On_Base)]), 4)
# 長打率：長打數 / 打數 -> (1B + 2B * 2 + 3B * 3 + HR * 4 + IH * 4) / At_Bat
right_vs_left_Total_Bases_Sum = right_pitcher_verus_left_hitter['result'].apply(lambda n: TB.get(n, 0)).sum()
right_vs_left_Slugging_Percentage = round(right_vs_left_Total_Bases_Sum / len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(At_Bat)]), 4)


# 純長打率
right_vs_left_Isolated_Power = round(right_vs_left_Slugging_Percentage - right_vs_left_Batting_Average, 4)
# 三振率
right_vs_left_Strikeout_Rate = round(len(right_pitcher_verus_left_hitter[right_pitcher_verus_left_hitter['result'] == 'SO']) / len(right_pitcher_verus_left_hitter), 4)
# 保送率
right_vs_left_Walk_Rate = round(len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'] == 'uBB',]) / len(right_pitcher_verus_left_hitter), 4)
# 場內安打率
right_vs_left_Batting_Aberage_on_Balls_In_play = round((len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(Hit)]) - len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'] == 'HR',]) )/ (len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'].isin(At_Bat)]) - len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'] == 'SO',]) - len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'] == 'HR',]) + len(right_pitcher_verus_left_hitter.loc[right_pitcher_verus_left_hitter['result'] == 'SF',])), 4)

right_vs_left_OPS = round(right_vs_left_On_Base_Percentage+ right_vs_left_Slugging_Percentage, 4)

print(f"右投對左打打擊率：{right_vs_left_Batting_Average}")
print(f"右投對左打上壘率：{right_vs_left_On_Base_Percentage}")
print(f"右投對左打長打率：{right_vs_left_Slugging_Percentage}")
print(f"右投對左打純長打率：{right_vs_left_Isolated_Power}")
print(f"右投對左打三振率：{right_vs_left_Strikeout_Rate}")
print(f"右投對左打保送率：{right_vs_left_Walk_Rate}")
print(f"右投對左打場內安打率：{right_vs_left_Batting_Aberage_on_Balls_In_play}")
print(f"右投對左打 OPS ：{right_vs_left_OPS}")


右投對左打打擊率：0.267
右投對左打上壘率：0.3361
右投對左打長打率：0.3817
右投對左打純長打率：0.1147
右投對左打三振率：0.1697
右投對左打保送率：0.0847
右投對左打場內安打率：0.3115
右投對左打 OPS ：0.7178


In [10]:
# 右投對右打
right_vs_right_Batting_Average = round(len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(Hit),]) / len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(At_Bat)]), 4)
right_vs_right_On_Base_Percentage = round(len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(On_Base)]) / len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(At_Bat_and_On_Base)]), 4)

right_vs_right_Total_Bases_Sum = right_pitcher_verus_right_hitter['result'].apply(lambda n: TB.get(n, 0)).sum()
right_vs_right_Slugging_Percentage = round(right_vs_right_Total_Bases_Sum / len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(At_Bat)]), 4)


# 純長打率
right_vs_right_Isolated_Power = round(right_vs_right_Slugging_Percentage - right_vs_right_Batting_Average, 4)
# 三振率
right_vs_right_Strikeout_Rate = round(len(right_pitcher_verus_right_hitter[right_pitcher_verus_right_hitter['result'] == 'SO']) / len(right_pitcher_verus_right_hitter), 4)
# 保送率
right_vs_right_Walk_Rate = round(len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'] == 'uBB',]) / len(right_pitcher_verus_right_hitter), 4)
# 場內安打率
right_vs_right_Batting_Aberage_on_Balls_In_play = round((len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(Hit)]) - len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'] == 'HR',]) )/ (len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'].isin(At_Bat)]) - len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'] == 'SO',]) - len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'] == 'HR',]) + len(right_pitcher_verus_right_hitter.loc[right_pitcher_verus_right_hitter['result'] == 'SF',])), 4)


right_vs_right_OPS = round(right_vs_right_On_Base_Percentage + right_vs_right_Slugging_Percentage, 4)

print(f"右投對右打打擊率：{right_vs_right_Batting_Average}")
print(f"右投對右打上壘率：{right_vs_right_On_Base_Percentage}")
print(f"右投對右打長打率：{right_vs_right_Slugging_Percentage}")
print(f"右投對右打純長打率：{right_vs_right_Isolated_Power}")
print(f"右投對右打三振率：{right_vs_right_Strikeout_Rate}")
print(f"右投對右打保送率：{right_vs_right_Walk_Rate}")
print(f"右投對右打場內安打率：{right_vs_right_Batting_Aberage_on_Balls_In_play}")
print(f"右投對右打 OPS ：{right_vs_right_OPS}")


右投對右打打擊率：0.2603
右投對右打上壘率：0.3223
右投對右打長打率：0.3476
右投對右打純長打率：0.0873
右投對右打三振率：0.1767
右投對右打保送率：0.0675
右投對右打場內安打率：0.311
右投對右打 OPS ：0.6699


## 各隊代打概況

In [23]:
# 整體代打次數
from tarfile import LinkOutsideDestinationError

from numpy._core.multiarray import dragon4_scientific


pinch_hitter = pa_record['isPH'].sum()
all_hitter = len(pa_record)
PH_rate = round(pinch_hitter / all_hitter, 4)
print(f"整體代打率：{pinch_hitter}/{all_hitter}: {PH_rate}")

# 整體使用相反慣用手打者次數
pinch_hitter_opposite_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] != pa_record['pitcherHand']), 'isPH'].sum()
print(f"整體使用相反慣用手打者比例：{pinch_hitter_opposite_hand}/{pinch_hitter}: {round(pinch_hitter_opposite_hand / pinch_hitter, 4)}")

# 派出左打克右投的比例
pinch_hitter_left_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] == 'L') & (pa_record['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'左打對右投: {pinch_hitter_left_hand}/{pinch_hitter_opposite_hand}: {round(pinch_hitter_left_hand / pinch_hitter_opposite_hand, 4)}')
# 派出右打克左投的比例
pinch_hitter_right_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] == 'R') & (pa_record['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'右打對左投: {pinch_hitter_right_hand}/{pinch_hitter_opposite_hand}: {round(pinch_hitter_right_hand / pinch_hitter_opposite_hand, 4)}')
print('-' * 10)

# 各隊代打次數
elephants = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='中信兄弟')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='中信兄弟'))]
lions = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='統一7-ELEVEn獅')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='統一7-ELEVEn獅'))]
monkeys = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='樂天桃猿')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='樂天桃猿'))]
dragons = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='味全龍')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='味全龍'))]
hawks = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='台鋼雄鷹')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='台鋼雄鷹'))]
guardians = pa_record.loc[((pa_record['team_type']=='home') & (pa_record['homeTeam']=='富邦悍將')) | ((pa_record['team_type']=='away') & (pa_record['awayTeam']=='富邦悍將'))]
pinch_hitter_elephants = elephants['isPH'].sum()
pinch_hitter_lions = lions['isPH'].sum()
pinch_hitter_monkeys = monkeys['isPH'].sum()
pinch_hitter_dragons = dragons['isPH'].sum()
pinch_hitter_hawks = hawks['isPH'].sum()
pinch_hitter_guardians = guardians['isPH'].sum()
print(f'中信兄弟代打次數: {pinch_hitter_elephants}')
print(f'統一7-ELEVEn獅代打次數: {pinch_hitter_lions}')
print(f'樂天桃猿代打次數: {pinch_hitter_monkeys}')
print(f'味全龍代打次數: {pinch_hitter_dragons}')
print(f'台鋼雄鷹代打次數: {pinch_hitter_hawks}')
print(f'富邦悍將代打次數: {pinch_hitter_guardians}')
print(f'總計代打次數：{pinch_hitter_elephants + pinch_hitter_lions + pinch_hitter_monkeys + pinch_hitter_dragons + pinch_hitter_hawks + pinch_hitter_guardians}')
print('-' * 10)
# 各隊使用使用相反慣用手打者次數
pinch_hitter_elephants_opposite_hand = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] != elephants['pitcherHand']), 'isPH'].sum()
pinch_hitter_lions_opposite_hand = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] != lions['pitcherHand']), 'isPH'].sum()
pinch_hitter_monkeys_opposite_hand = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] != monkeys['pitcherHand']), 'isPH'].sum()
pinch_hitter_dragons_opposite_hand = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] != dragons['pitcherHand']), 'isPH'].sum()
pinch_hitter_hawks_opposite_hand = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] != hawks['pitcherHand']), 'isPH'].sum()
pinch_hitter_guardians_opposite_hand = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] != guardians['pitcherHand']), 'isPH'].sum()
print(f'中信兄弟使用相反慣用手代打比例: {pinch_hitter_elephants_opposite_hand}/{pinch_hitter_elephants}: {round(pinch_hitter_elephants_opposite_hand / pinch_hitter_elephants, 4)}')
print(f'統一7-ELEVEn獅使用相反慣用手代打比例: {pinch_hitter_lions_opposite_hand}/{pinch_hitter_lions}: {round(pinch_hitter_lions_opposite_hand / pinch_hitter_lions, 4)}')
print(f'樂天桃猿使用相反慣用手代打比例: {pinch_hitter_monkeys_opposite_hand}/{pinch_hitter_monkeys}: {round(pinch_hitter_monkeys_opposite_hand / pinch_hitter_monkeys, 4)}')
print(f'味全龍使用相反慣用手代打比例: {pinch_hitter_dragons_opposite_hand}/{pinch_hitter_dragons}: {round(pinch_hitter_dragons_opposite_hand / pinch_hitter_dragons, 4)}')
print(f'台鋼雄鷹使用相反慣用手代打比例: {pinch_hitter_hawks_opposite_hand}/{pinch_hitter_hawks}: {round(pinch_hitter_hawks_opposite_hand / pinch_hitter_hawks, 4)}')
print(f'富邦悍將使用相反慣用手代打比例: {pinch_hitter_guardians_opposite_hand}/{pinch_hitter_guardians}: {round(pinch_hitter_guardians_opposite_hand / pinch_hitter_guardians, 4)}')







整體代打率：520/27600: 0.0188
整體使用相反慣用手打者比例：342/520: 0.6577
左打對右投: 242/342: 0.7076
右打對左投: 100/342: 0.2924
----------
中信兄弟代打次數: 95
統一7-ELEVEn獅代打次數: 99
樂天桃猿代打次數: 79
味全龍代打次數: 73
台鋼雄鷹代打次數: 80
富邦悍將代打次數: 94
總計代打次數：520
----------
中信兄弟使用相反慣用手代打比例: 67/95: 0.7053
統一7-ELEVEn獅使用相反慣用手代打比例: 64/99: 0.6465
樂天桃猿使用相反慣用手代打比例: 38/79: 0.481
味全龍使用相反慣用手代打比例: 59/73: 0.8082
台鋼雄鷹使用相反慣用手代打比例: 53/80: 0.6625
富邦悍將使用相反慣用手代打比例: 61/94: 0.6489


In [26]:
# 中信兄弟派出左打克右投的比例
pinch_hitter_left_hand_elephants = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] == 'L') & (elephants['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'中信兄弟派出左打克右投的比例: {pinch_hitter_left_hand_elephants}/{pinch_hitter_elephants_opposite_hand}: {round(pinch_hitter_left_hand_elephants / pinch_hitter_elephants_opposite_hand, 4)}')
# 中信兄弟派出右打克左投的比例
pinch_hitter_right_hand_elephants = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] == 'R') & (elephants['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'中信兄弟派出右打克左投的比例: {pinch_hitter_right_hand_elephants}/{pinch_hitter_elephants_opposite_hand}: {round(pinch_hitter_right_hand_elephants / pinch_hitter_elephants_opposite_hand, 4)}')
# 統一7-ELEVEn獅派出左打克右投的比例
pinch_hitter_left_hand_lions = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] == 'L') & (lions['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'統一7-ELEVEn獅派出左打克右投的比例: {pinch_hitter_left_hand_lions}/{pinch_hitter_lions_opposite_hand}: {round(pinch_hitter_left_hand_lions / pinch_hitter_lions_opposite_hand, 4)}')
# 統一7-ELEVEn獅派出右打克左投的比例
pinch_hitter_right_hand_lions = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] == 'R') & (lions['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'統一7-ELEVEn獅派出右打克左投的比例: {pinch_hitter_right_hand_lions}/{pinch_hitter_lions_opposite_hand}: {round(pinch_hitter_right_hand_lions / pinch_hitter_lions_opposite_hand, 4)}')

# 樂天桃猿派出左打克右投的比例
pinch_hitter_left_hand_monkeys = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] == 'L') & (monkeys['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'樂天桃猿派出左打克右投的比例: {pinch_hitter_left_hand_monkeys}/{pinch_hitter_monkeys_opposite_hand}: {round(pinch_hitter_left_hand_monkeys / pinch_hitter_monkeys_opposite_hand, 4)}')
# 樂天桃猿派出右打克左投的比例
pinch_hitter_right_hand_monkeys = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] == 'R') & (monkeys['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'樂天桃猿派出右打克左投的比例: {pinch_hitter_right_hand_monkeys}/{pinch_hitter_monkeys_opposite_hand}: {round(pinch_hitter_right_hand_monkeys / pinch_hitter_monkeys_opposite_hand, 4)}')



中信兄弟派出左打克右投的比例: 54/67: 0.806
中信兄弟派出右打克左投的比例: 13/67: 0.194
統一7-ELEVEn獅派出左打克右投的比例: 50/64: 0.7812
統一7-ELEVEn獅派出右打克左投的比例: 14/64: 0.2188
樂天桃猿派出左打克右投的比例: 17/38: 0.4474
樂天桃猿派出右打克左投的比例: 21/38: 0.5526


In [27]:
# 味全龍派出左打克右投的比例
pinch_hitter_left_hand_dragons = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] == 'L') & (dragons['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'味全龍派出左打克右投的比例: {pinch_hitter_left_hand_dragons}/{pinch_hitter_dragons_opposite_hand}: {round(pinch_hitter_left_hand_dragons / pinch_hitter_dragons_opposite_hand, 4)}')
# 味全龍派出右打克左投的比例
pinch_hitter_right_hand_dragons = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] == 'R') & (dragons['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'味全龍派出右打克左投的比例: {pinch_hitter_right_hand_dragons}/{pinch_hitter_dragons_opposite_hand}: {round(pinch_hitter_right_hand_dragons / pinch_hitter_dragons_opposite_hand, 4)}')

# 台鋼雄鷹派出左打克右投的比例
pinch_hitter_left_hand_hawks = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] == 'L') & (hawks['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'台鋼雄鷹派出左打克右投的比例: {pinch_hitter_left_hand_hawks}/{pinch_hitter_hawks_opposite_hand}: {round(pinch_hitter_left_hand_hawks / pinch_hitter_hawks_opposite_hand, 4)}')
# 台鋼雄鷹派出右打克左投的比例
pinch_hitter_right_hand_hawks = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] == 'R') & (hawks['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'台鋼雄鷹派出右打克左投的比例: {pinch_hitter_right_hand_hawks}/{pinch_hitter_hawks_opposite_hand}: {round(pinch_hitter_right_hand_hawks / pinch_hitter_hawks_opposite_hand, 4)}')

# 富邦悍將派出左打克右投的比例
pinch_hitter_left_hand_guardians = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] == 'L') & (guardians['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'富邦悍將派出左打克右投的比例: {pinch_hitter_left_hand_guardians}/{pinch_hitter_guardians_opposite_hand}: {round(pinch_hitter_left_hand_guardians / pinch_hitter_guardians_opposite_hand, 4)}')
# 富邦悍將派出右打克左投的比例
pinch_hitter_right_hand_guardians = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] == 'R') & (guardians['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'富邦悍將派出右打克左投的比例: {pinch_hitter_right_hand_guardians}/{pinch_hitter_guardians_opposite_hand}: {round(pinch_hitter_right_hand_guardians / pinch_hitter_guardians_opposite_hand, 4)}')




味全龍派出左打克右投的比例: 37/59: 0.6271
味全龍派出右打克左投的比例: 22/59: 0.3729
台鋼雄鷹派出左打克右投的比例: 41/53: 0.7736
台鋼雄鷹派出右打克左投的比例: 12/53: 0.2264
富邦悍將派出左打克右投的比例: 43/61: 0.7049
富邦悍將派出右打克左投的比例: 18/61: 0.2951


In [30]:
# 整體使用相同慣用手打者比例
pinch_hitter_same_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] == pa_record['pitcherHand']), 'isPH'].sum()
print(f'整體使用相同慣用手打者比例: {pinch_hitter_same_hand}/{pinch_hitter}: {round(pinch_hitter_same_hand / pinch_hitter, 4)}')
# 左打對左投
pinch_hitter_same_left_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] == 'L') & (pa_record['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'左打對左投: {pinch_hitter_same_left_hand}/{pinch_hitter_same_hand}: {round(pinch_hitter_same_left_hand / pinch_hitter_same_hand, 4)}')
# 右打對右投
pinch_hitter_same_right_hand = pa_record.loc[(pa_record['isPH'] == True) & (pa_record['batterHand'] == 'R') & (pa_record['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'右打對右投: {pinch_hitter_same_right_hand}/{pinch_hitter_same_hand}: {round(pinch_hitter_same_right_hand / pinch_hitter_same_hand, 4)}')
print('-' * 10)
# 各隊使用使用相同慣用手打者次數
pinch_hitter_same_hand_elephants = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] == elephants['pitcherHand']), 'isPH'].sum()
print(f'中信兄弟使用相同慣用手代打比例: {pinch_hitter_same_hand_elephants}/{pinch_hitter_elephants}: {round(pinch_hitter_same_hand_elephants / pinch_hitter_elephants, 4)}')
pinch_hitter_same_hand_lions = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] == lions['pitcherHand']), 'isPH'].sum()
print(f'統一7-ELEVEn獅使用相同慣用手代打比例: {pinch_hitter_same_hand_lions}/{pinch_hitter_lions}: {round(pinch_hitter_same_hand_lions / pinch_hitter_lions, 4)}')
pinch_hitter_same_hand_monkeys = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] == monkeys['pitcherHand']), 'isPH'].sum()
print(f'樂天桃猿使用相同慣用手代打比例: {pinch_hitter_same_hand_monkeys}/{pinch_hitter_monkeys}: {round(pinch_hitter_same_hand_monkeys / pinch_hitter_monkeys, 4)}')
pinch_hitter_same_hand_dragons = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] == dragons['pitcherHand']), 'isPH'].sum()
print(f'味全龍使用相同慣用手代打比例: {pinch_hitter_same_hand_dragons}/{pinch_hitter_dragons}: {round(pinch_hitter_same_hand_dragons / pinch_hitter_dragons, 4)}')
pinch_hitter_same_hand_hawks = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] == hawks['pitcherHand']), 'isPH'].sum()
print(f'台鋼雄鷹使用相同慣用手代打比例: {pinch_hitter_same_hand_hawks}/{pinch_hitter_hawks}: {round(pinch_hitter_same_hand_hawks / pinch_hitter_hawks, 4)}')
pinch_hitter_same_hand_guardians = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] == guardians['pitcherHand']), 'isPH'].sum()
print(f'富邦悍將使用相同慣用手代打比例: {pinch_hitter_same_hand_guardians}/{pinch_hitter_guardians}: {round(pinch_hitter_same_hand_guardians / pinch_hitter_guardians, 4)}')



整體使用相同慣用手打者比例: 178/520: 0.3423
左打對左投: 26/178: 0.1461
右打對右投: 152/178: 0.8539
----------
中信兄弟使用相同慣用手代打比例: 28/95: 0.2947
統一7-ELEVEn獅使用相同慣用手代打比例: 35/99: 0.3535
樂天桃猿使用相同慣用手代打比例: 41/79: 0.519
味全龍使用相同慣用手代打比例: 14/73: 0.1918
台鋼雄鷹使用相同慣用手代打比例: 27/80: 0.3375
富邦悍將使用相同慣用手代打比例: 33/94: 0.3511


In [31]:
# 中信兄弟派出左打對左投的比例
pinch_hitter_same_left_hand_elephants = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] == 'L') & (elephants['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'中信兄弟派出左打對左投的比例: {pinch_hitter_same_left_hand_elephants}/{pinch_hitter_same_hand_elephants}: {round(pinch_hitter_same_left_hand_elephants / pinch_hitter_same_hand_elephants, 4)}')
# 中信兄弟派出右打對右投的比例
pinch_hitter_same_right_hand_elephants = elephants.loc[(elephants['isPH'] == True) & (elephants['batterHand'] == 'R') & (elephants['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'中信兄弟派出右打對右投的比例: {pinch_hitter_same_right_hand_elephants}/{pinch_hitter_same_hand_elephants}: {round(pinch_hitter_same_right_hand_elephants / pinch_hitter_same_hand_elephants, 4)}')

# 統一7-ELEVEn獅派出左打對左投的比例
pinch_hitter_same_left_hand_lions = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] == 'L') & (lions['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'統一7-ELEVEn獅派出左打對左投的比例: {pinch_hitter_same_left_hand_lions}/{pinch_hitter_same_hand_lions}: {round(pinch_hitter_same_left_hand_lions / pinch_hitter_same_hand_lions, 4)}')
# 統一7-ELEVEn獅派出右打對右投的比例
pinch_hitter_same_right_hand_lions = lions.loc[(lions['isPH'] == True) & (lions['batterHand'] == 'R') & (lions['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'統一7-ELEVEn獅派出右打對右投的比例: {pinch_hitter_same_right_hand_lions}/{pinch_hitter_same_hand_lions}: {round(pinch_hitter_same_right_hand_lions / pinch_hitter_same_hand_lions, 4)}')

# 樂天桃猿派出左打對左投的比例
pinch_hitter_same_left_hand_monkeys = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] == 'L') & (monkeys['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'樂天桃猿派出左打對左投的比例: {pinch_hitter_same_left_hand_monkeys}/{pinch_hitter_same_hand_monkeys}: {round(pinch_hitter_same_left_hand_monkeys / pinch_hitter_same_hand_monkeys, 4)}')
# 樂天桃猿派出右打對右投的比例
pinch_hitter_same_right_hand_monkeys = monkeys.loc[(monkeys['isPH'] == True) & (monkeys['batterHand'] == 'R') & (monkeys['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'樂天桃猿派出右打對右投的比例: {pinch_hitter_same_right_hand_monkeys}/{pinch_hitter_same_hand_monkeys}: {round(pinch_hitter_same_right_hand_monkeys / pinch_hitter_same_hand_monkeys, 4)}')


中信兄弟派出左打對左投的比例: 8/28: 0.2857
中信兄弟派出右打對右投的比例: 20/28: 0.7143
統一7-ELEVEn獅派出左打對左投的比例: 3/35: 0.0857
統一7-ELEVEn獅派出右打對右投的比例: 32/35: 0.9143
樂天桃猿派出左打對左投的比例: 2/41: 0.0488
樂天桃猿派出右打對右投的比例: 39/41: 0.9512


In [36]:
# 味全龍派出左打對左投的比例
pinch_hitter_same_left_hand_dragons = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] == 'L') & (dragons['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'味全龍派出左打對左投的比例: {pinch_hitter_same_left_hand_dragons}/{pinch_hitter_same_hand_dragons}: {round(pinch_hitter_same_left_hand_dragons / pinch_hitter_same_hand_dragons, 4)}')
# 味全龍派出右打對右投的比例
pinch_hitter_same_right_hand_dragons = dragons.loc[(dragons['isPH'] == True) & (dragons['batterHand'] == 'R') & (dragons['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'味全龍派出右打對右投的比例: {pinch_hitter_same_right_hand_dragons}/{pinch_hitter_same_hand_dragons}: {round(pinch_hitter_same_right_hand_dragons / pinch_hitter_same_hand_dragons, 4)}')

# 台鋼雄鷹派出左打對左投的比例
pinch_hitter_same_left_hand_hawks = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] == 'L') & (hawks['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'台鋼雄鷹派出左打對左投的比例: {pinch_hitter_same_left_hand_hawks}/{pinch_hitter_same_hand_hawks}: {round(pinch_hitter_same_left_hand_hawks / pinch_hitter_same_hand_hawks, 4)}')
# 台鋼雄鷹派出右打對右投的比例
pinch_hitter_same_right_hand_hawks = hawks.loc[(hawks['isPH'] == True) & (hawks['batterHand'] == 'R') & (hawks['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'台鋼雄鷹派出右打對右投的比例: {pinch_hitter_same_right_hand_hawks}/{pinch_hitter_same_hand_hawks}: {round(pinch_hitter_same_right_hand_hawks / pinch_hitter_same_hand_hawks, 4)}')

# 富邦悍將派出左打對左投的比例
pinch_hitter_same_left_hand_guardians = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] == 'L') & (guardians['pitcherHand'] == 'L'), 'isPH'].sum()
print(f'富邦悍將派出左打對左投的比例: {pinch_hitter_same_left_hand_guardians}/{pinch_hitter_same_hand_guardians}: {round(pinch_hitter_same_left_hand_guardians / pinch_hitter_same_hand_guardians, 4)}')
# 富邦悍將派出右打對右投的比例
pinch_hitter_same_right_hand_guardians = guardians.loc[(guardians['isPH'] == True) & (guardians['batterHand'] == 'R') & (guardians['pitcherHand'] == 'R'), 'isPH'].sum()
print(f'富邦悍將派出右打對右投的比例: {pinch_hitter_same_right_hand_guardians}/{pinch_hitter_same_hand_guardians}: {round(pinch_hitter_same_right_hand_guardians / pinch_hitter_same_hand_guardians, 4)}')


味全龍派出左打對左投的比例: 1/14: 0.0714
味全龍派出右打對右投的比例: 13/14: 0.9286
台鋼雄鷹派出左打對左投的比例: 3/27: 0.1111
台鋼雄鷹派出右打對右投的比例: 24/27: 0.8889
富邦悍將派出左打對左投的比例: 9/33: 0.2727
富邦悍將派出右打對右投的比例: 24/33: 0.7273
